# 第 5 章：GroupBy、KPI、資料合併與客群分析

本 Notebook 融合 `lesson05.ipynb` 與 `practice_ch05.py`，並修復原 Notebook 中的中文編碼亂碼。

**適合對象**：已具備 Pandas 篩選、日期轉換與基本 DataFrame 操作能力的學習者。

**學習目標**：
- 將訂單品項彙總成訂單層級營收。
- 使用 `groupby()` 與命名聚合計算 KPI。
- 理解訂單數、總營收與平均訂單金額（AOV）。
- 在合併客戶資料前後驗證列數與鍵值唯一性。
- 分析不同客群的訂單、客戶、營收與營收占比。
- 使用 `pivot_table()` 建立取得管道與客群交叉表。

## 學習流程

1. 載入資料並確認資料粒度
2. 計算訂單品項營收
3. 彙總成訂單層級分析表
4. 計算整體 KPI
5. 依付款方式彙總 KPI
6. 合併客戶資料並驗證結果
7. 進行客群分析
8. 建立取得管道 × 客群交叉表
9. 驗證分組加總與來源資料一致

## 1. 環境設定與資料載入

使用 `common.py` 的共用函式載入資料。雖然共用模組已提供 `order_facts()`，本章前半段仍會手動建立一次，幫助理解其中的資料處理步驟。

In [1]:
import pandas as pd
from IPython.display import display

from common import ensure_packages, load_data, order_facts

ensure_packages()
data = load_data()

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

customers = data["customers"]
orders = data["orders"]
order_items = data["order_items"]

print(f"customers：{len(customers):,} 列")
print(f"orders：{len(orders):,} 列")
print(f"order_items：{len(order_items):,} 列")

customers：2,500 列
orders：22,000 列
order_items：39,627 列


## 2. 先理解資料粒度

「資料粒度」表示一列資料代表什麼：

- `customers`：一列代表一位客戶。
- `orders`：一列代表一張訂單。
- `order_items`：一列代表訂單中的一個商品品項。

同一張訂單可能有多個商品，因此 `order_items` 的列數通常多於 `orders`。計算訂單 KPI 前，必須先把品項彙總到訂單層級。

In [2]:
grain_summary = pd.DataFrame({
    "資料表": ["customers", "orders", "order_items"],
    "列數": [len(customers), len(orders), len(order_items)],
    "主要鍵值的唯一數": [
        customers["customer_id"].nunique(),
        orders["order_id"].nunique(),
        order_items["order_id"].nunique(),
    ],
})
display(grain_summary)

,資料表,列數,主要鍵值的唯一數
0,customers,2500,2500
1,orders,22000,22000
2,order_items,39627,22000


## 3. 計算訂單品項營收

每筆品項的折扣後營收公式為：

`數量 × 單價 × (1 - 折扣率)`

使用 `.copy()` 保留原始資料，再新增 `line_revenue` 欄位。

In [3]:
items = order_items.copy()
items["line_revenue"] = (
    items["quantity"]
    * items["unit_price"]
    * (1 - items["discount_rate"])
)

display(items.head())

,order_id,product_id,quantity,unit_price,discount_rate,line_revenue
0,1,28,1,567,0.05,538.65
1,2,2,1,3850,0.05,"3,657.50"
2,2,18,1,778,0.10,700.20
3,3,19,1,2193,0.00,"2,193.00"
4,3,36,1,1580,0.15,"1,343.00"


## 4. 彙總成訂單營收

將相同 `order_id` 的品項營收加總後，一列才代表一張訂單。`as_index=False` 讓 `order_id` 保持為一般欄位，而不是變成索引。

In [4]:
order_revenue = (
    items.groupby("order_id", as_index=False)
    .agg(line_revenue=("line_revenue", "sum"))
)

print(f"品項資料：{len(items):,} 列")
print(f"訂單營收資料：{len(order_revenue):,} 列")
print("order_id 是否唯一：", order_revenue["order_id"].is_unique)
display(order_revenue.head())

品項資料：39,627 列
訂單營收資料：22,000 列
order_id 是否唯一： True


,order_id,line_revenue
0,1,538.65
1,2,"4,357.70"
2,3,"3,536.00"
3,4,"1,193.40"
4,5,"13,175.50"


## 5. 合併訂單狀態並保留完成訂單

營收表只有 `order_id` 與金額，需要合併 `orders` 才能取得客戶、日期、狀態及付款方式。完成訂單的營收才納入本章 KPI。

In [5]:
order_level = order_revenue.merge(
    orders, on="order_id", how="left", validate="one_to_one"
)
completed = order_level.loc[
    order_level["status"] == "completed"
].copy()
completed["order_date"] = pd.to_datetime(completed["order_date"])

print(f"全部訂單：{len(order_level):,} 筆")
print(f"完成訂單：{len(completed):,} 筆")
display(completed.head())

全部訂單：22,000 筆
完成訂單：20,413 筆


,order_id,line_revenue,customer_id,order_date,status,payment_type
0,1,538.65,2323,2025-05-02,completed,wallet
1,2,"4,357.70",117,2024-07-14,completed,wallet
3,4,"1,193.40",1240,2024-07-08,completed,atm
4,5,"13,175.50",714,2024-08-12,completed,atm
5,6,"3,657.50",2005,2025-01-14,completed,card


### `validate="one_to_one"` 的用途

這項參數要求左右兩邊的 `order_id` 都必須唯一。若資料不符合一對一關係，Pandas 會直接報錯，避免不小心造成列數膨脹。

## 6. 計算整體 KPI

本章使用三個常見指標：

- 完成訂單數：不重複的 `order_id` 數量。
- 總營收：完成訂單營收加總。
- AOV（Average Order Value）：總營收 ÷ 完成訂單數。

In [6]:
total_revenue = completed["line_revenue"].sum()
completed_orders = completed["order_id"].nunique()
aov = total_revenue / completed_orders

overall_kpi = pd.DataFrame({
    "指標": ["完成訂單數", "總營收", "平均訂單金額（AOV）"],
    "數值": [completed_orders, total_revenue, aov],
})
display(overall_kpi)

,指標,數值
0,完成訂單數,"20,413.00"
1,總營收,"101,771,799.75"
2,平均訂單金額（AOV）,"4,985.64"


## 7. 依付款方式彙總 KPI

`groupby("payment_type")` 會將相同付款方式的訂單放入同一組，再使用命名聚合一次計算多個指標。

In [7]:
payment_kpi = (
    completed.groupby("payment_type", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        revenue=("line_revenue", "sum"),
        aov=("line_revenue", "mean"),
    )
    .sort_values("revenue", ascending=False)
)

display(payment_kpi.round(2))

,payment_type,orders,revenue,aov
1,card,10166,"50,567,692.45","4,974.20"
0,atm,5111,"25,452,594.55","4,979.96"
3,wallet,3076,"15,343,032.20","4,987.98"
2,cod,2060,"10,408,480.55","5,052.66"


因為 `completed` 已是一列一張訂單，所以 `line_revenue.mean()` 等於 AOV。若資料仍在品項層級，直接取平均就會變成「平均品項營收」，不是平均訂單金額。

## 8. 使用共用函式建立分析表

`common.order_facts()` 封裝了前面的品項營收、訂單彙總、合併與完成訂單篩選。先與手動結果核對，確保兩者口徑一致。

In [8]:
facts = order_facts(data).copy()

comparison = pd.DataFrame({
    "建立方式": ["手動建立", "order_facts()"],
    "列數": [len(completed), len(facts)],
    "營收加總": [completed["line_revenue"].sum(), facts["line_revenue"].sum()],
})
display(comparison)

assert len(completed) == len(facts), "兩種建立方式的列數不一致"
assert abs(completed["line_revenue"].sum() - facts["line_revenue"].sum()) < 0.01

,建立方式,列數,營收加總
0,手動建立,20413,"101,771,799.75"
1,order_facts(),20413,"101,771,799.75"


## 9. 合併客戶資料前先驗證鍵值

若 `customers.customer_id` 重複，一張訂單可能在合併後複製成多列。先確認客戶鍵值唯一，再使用 `validate="many_to_one"` 表明關係：多張訂單可以屬於同一位客戶，但客戶表中的每位客戶只能有一列。

In [9]:
print(f"facts 合併前列數：{len(facts):,}")
print("customers.customer_id 是否唯一：", customers["customer_id"].is_unique)

merged = facts.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
)

print(f"合併後列數：{len(merged):,}")
print(f"segment 缺失筆數：{merged['segment'].isna().sum():,}")

assert len(merged) == len(facts), "合併後列數與原訂單數不一致"

facts 合併前列數：20,413
customers.customer_id 是否唯一： True
合併後列數：20,413
segment 缺失筆數：0


### 為什麼使用左合併？

`how="left"` 會保留所有完成訂單。若某個 `customer_id` 在客戶表找不到，訂單仍會保留，但客群欄位會是缺失值，讓問題可被發現。若使用內合併，找不到客戶的訂單會直接消失。

## 10. 客群 KPI 分析

依 `segment` 分組，計算：

- `order_count`：訂單數。
- `unique_customers`：不重複客戶數。
- `total_revenue`：總營收。
- `avg_order_value`：平均訂單金額。
- `revenue_share`：該客群營收占全部客群營收的百分比。

In [10]:
segment_analysis = (
    merged.groupby("segment", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        total_revenue=("line_revenue", "sum"),
        avg_order_value=("line_revenue", "mean"),
    )
)
segment_analysis["revenue_share"] = (
    segment_analysis["total_revenue"]
    / segment_analysis["total_revenue"].sum()
    * 100
)
segment_analysis = segment_analysis.sort_values(
    "total_revenue", ascending=False
)

display(segment_analysis.round(2))

,segment,order_count,unique_customers,total_revenue,avg_order_value,revenue_share
1,new,12177,1490,"60,270,109.20","4,949.50",59.22
0,growth,6155,753,"30,862,612.10","5,014.23",30.33
2,vip,2081,256,"10,639,078.45","5,112.48",10.45


營收占比的總和應接近 100%。因顯示時四捨五入，畫面上可能出現極小誤差。

## 11. 取得管道 × 客群交叉表

`pivot_table()` 可以同時以列和欄分類。這裡：

- 列：客戶取得管道 `acquisition_channel`。
- 欄：客群 `segment`。
- 值：訂單營收 `line_revenue`。
- 統計：訂單筆數 `count` 與平均訂單金額 `mean`。
- `margins=True`：加入 `All` 總計列與總計欄。

In [11]:
channel_segment_cross = merged.pivot_table(
    values="line_revenue",
    index="acquisition_channel",
    columns="segment",
    aggfunc=["count", "mean"],
    margins=True,
)

display(channel_segment_cross.round(2))

count                         mean                    \
segment             growth    new   vip    All   growth      new      vip   
acquisition_channel                                                         
ads                   1194   2640   403   4237 4,871.73 4,907.97 5,046.19   
organic               1163   2532   391   4086 5,134.43 4,898.96 5,420.78   
partner               1313   2389   465   4167 5,070.05 5,002.18 4,743.32   
referral              1262   2520   425   4207 4,981.61 4,967.69 5,173.21   
social                1223   2096   397   3716 5,012.80 4,980.98 5,243.53   
All                   6155  12177  2081  20413 5,014.23 4,949.50 5,112.48   

                              
segment                  All  
acquisition_channel           
ads                 4,910.90  
organic             5,015.92  
partner             4,994.68  
referral            4,992.62  
social              5,019.50  
All                 4,985.64

這張表使用多層欄位：最上層是統計方法 `count`／`mean`，下一層是客群。閱讀時應先確認目前看的指標層級。

## 12. 分組結果完整性檢查

客群分組後的訂單數與營收加總，應回到合併前的來源總數。這能協助發現合併遺失、重複或缺失客群未納入分組等問題。

In [12]:
segment_order_total = int(segment_analysis["order_count"].sum())
segment_revenue_total = segment_analysis["total_revenue"].sum()
source_order_total = len(merged)
source_revenue_total = merged["line_revenue"].sum()

validation = pd.DataFrame({
    "檢查項目": ["訂單數", "總營收", "營收占比"],
    "分組結果": [
        segment_order_total,
        segment_revenue_total,
        segment_analysis["revenue_share"].sum(),
    ],
    "預期結果": [source_order_total, source_revenue_total, 100.0],
})
validation["差異"] = validation["分組結果"] - validation["預期結果"]
display(validation)

,檢查項目,分組結果,預期結果,差異
0,訂單數,"20,413.00","20,413.00",0.00
1,總營收,"101,771,799.75","101,771,799.75",0.00
2,營收占比,100.00,100.00,0.00


## 13. 練習題

請依 `acquisition_channel` 計算訂單數、客戶數、總營收、AOV 與營收占比，並找出：

1. 營收最高的取得管道。
2. AOV 最高的取得管道。
3. 兩者是否為同一管道？

In [13]:
# TODO：可先遮住以下參考答案，再自行完成。
channel_analysis = (
    merged.groupby("acquisition_channel", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        total_revenue=("line_revenue", "sum"),
        avg_order_value=("line_revenue", "mean"),
    )
)
channel_analysis["revenue_share"] = (
    channel_analysis["total_revenue"]
    / channel_analysis["total_revenue"].sum()
    * 100
)

display(channel_analysis.sort_values("total_revenue", ascending=False).round(2))

,acquisition_channel,order_count,unique_customers,total_revenue,avg_order_value,revenue_share
3,referral,4207,512,"21,003,971.20","4,992.62",20.64
2,partner,4167,511,"20,812,828.40","4,994.68",20.45
0,ads,4237,511,"20,807,499.60","4,910.90",20.45
1,organic,4086,510,"20,495,033.90","5,015.92",20.14
4,social,3716,455,"18,652,466.65","5,019.50",18.33


## 常見錯誤與延伸

**常見錯誤**：
- 在品項層級直接計算訂單 AOV，導致指標實際上是平均品項營收。
- 合併前未檢查鍵值唯一性，造成列數膨脹。
- 使用內合併後未檢查遺失訂單。
- 把 `count()` 與 `nunique()` 混淆；前者算非缺失列數，後者算不重複值。
- 分組欄位含缺失值時，預設 `groupby()` 不會建立缺失值群組。

**延伸練習**：
- 加入每位客戶平均訂單數與平均客戶營收。
- 將客群 KPI 按月份拆分，觀察客群趨勢。
- 將樞紐表的多層欄位攤平成單層名稱，方便輸出 CSV。

## 重點整理

- 分析 KPI 前，必須先確認一列資料代表的粒度。
- `groupby()` 搭配命名聚合可一次產生多個清楚的指標。
- AOV 應在訂單層級計算。
- 合併時應驗證鍵值關係、列數與缺失情況。
- `pivot_table()` 適合建立兩個類別維度的交叉分析。